# 10 — The evaporating Page curve (v1.1 patch)

**Spacetime Lab v1.1 — the bell-shaped Page curve**

Phase 9 (the v1.0 milestone) shipped the **eternal BTZ** Page curve: the trivial Hartman-Maldacena saddle eventually crosses the constant island saddle at $2 S_{BH}$, the curve rises and *saturates*. That was enough to demonstrate the **qualitative resolution** of the Hawking information paradox in the cleanest possible analytical setting.

But an eternal black hole does not actually evaporate. The full Page curve — the one Page predicted in 1993 and the one the island formula was designed to reproduce — has a **bell shape**: rises, peaks, and *falls back to zero* when the BH has fully evaporated. That return to zero is what makes the toy model unitary.

This v1.1 patch ships the bell-shaped curve.

---

## What's new vs Phase 9

| | Phase 9 (`island.py`) | v1.1 (`evaporating.py`) |
|---|---|---|
| Background BH | Eternal BTZ (two-sided thermofield double) | Evaporating 4D Schwarzschild |
| Trivial saddle | Hartman-Maldacena $\frac{c}{3}\log\cosh\frac{2\pi t}{\beta}$ | $S_{BH}(0) - S_{BH}(t)$ |
| Island saddle | Constant $2 S_{BH}$ | Shrinking $S_{BH}(t) = 4\pi M(t)^2$ |
| Late-time behaviour | Saturates at $2 S_{BH}$ | **Falls back to 0 at $t_{evap}$** |
| Page time | Numerical brentq on transcendental | **Closed form** $t_P = (1 - \sqrt{2}/4)\,t_{evap}$ |

Both versions live in the same `spacetime_lab.holography` subpackage and both pass bit-exact gate tests.


## 1. The Schwarzschild evaporation law (Page 1976)

A 4D Schwarzschild BH has Hawking temperature $T_H = 1/(8\pi M)$ and emits Stefan-Boltzmann power $P \propto T^4 A \propto 1/M^2$. So $dM/dt \propto -1/M^2$, integrating to:

$$M(t) = M_0 \,(1 - t/t_{evap})^{1/3}, \qquad t_{evap} = \frac{5120\,\pi\,G^2 M_0^3}{\hbar c^4}.$$

In geometric units $G = c = \hbar = 1$ this becomes $t_{evap} = 5120\pi M_0^3$. The cubic shrinking law is the source of the bell curve's asymmetry: the BH loses entropy *faster than linearly* in time, so the Page time sits past the temporal midpoint of evaporation.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.holography import (
    schwarzschild_evaporation_time,
    schwarzschild_mass,
    bekenstein_hawking_entropy,
    hawking_saddle_entropy,
    island_saddle_entropy_evaporating,
    page_curve_evaporating,
    page_time_evaporating,
    page_time_evaporating_numerical,
    verify_evaporating_unitarity,
)

M0 = 1.0
t_evap = schwarzschild_evaporation_time(M0)
S_BH_0 = bekenstein_hawking_entropy(M0)

print(f't_evap         = {t_evap:.4f}    (= 5120 π M₀³ in geometric units)')
print(f'S_BH(0)        = {S_BH_0:.6f}    (= 4π M₀²)')
print(f'S_BH(0) / 2    = {S_BH_0/2:.6f}    (this is the predicted maximum of the Page curve)')

In [ ]:
# Plot the cubic shrinking law M(t)
ts = np.linspace(0, t_evap, 500)
Ms = np.array([schwarzschild_mass(t, M0) for t in ts])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ts / t_evap, Ms / M0, lw=2, color='#1f77b4')
ax.set_xlabel(r'$t / t_{evap}$')
ax.set_ylabel(r'$M(t) / M_0$')
ax.set_title(r'Schwarzschild evaporation: $M(t) = M_0(1 - t/t_{evap})^{1/3}$')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 2. The two saddles

The radiation entropy in our toy model has two competing contributions:

**No-island (Hawking) saddle** — the radiation, treated thermally, picks up the entropy lost by the BH:
$$S_H(t) = S_{BH}(0) - S_{BH}(t)$$
Monotone increasing from $0$ to $S_{BH}(0)$. *This is what would keep growing forever in Hawking's original calculation.*

**Island/QES saddle** — the quantum extremal surface contributes the *current* horizon area entropy:
$$S_\text{island}(t) = \frac{A(t)}{4 G_N} = S_{BH}(t)$$
Monotone *decreasing* from $S_{BH}(0)$ to $0$.

The two are **complementary**: $S_H(t) + S_\text{island}(t) = S_{BH}(0)$ at every instant.

In [ ]:
S_H_arr = np.array([hawking_saddle_entropy(t, M0) for t in ts])
S_I_arr = np.array([island_saddle_entropy_evaporating(t, M0) for t in ts])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ts / t_evap, S_H_arr, lw=2, color='#d62728', label=r'$S_H(t)$ (Hawking saddle)')
ax.plot(ts / t_evap, S_I_arr, lw=2, color='#2ca02c', label=r'$S_\mathrm{island}(t)$ (QES saddle)')
ax.axhline(S_BH_0, color='gray', ls='--', lw=1, alpha=0.6, label=r'$S_{BH}(0)$')
ax.set_xlabel(r'$t / t_{evap}$')
ax.set_ylabel('entropy')
ax.set_title('The two competing saddles')
ax.legend(loc='center right')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

# Sanity: complementarity at any t
for f in [0.1, 0.3, 0.5, 0.7, 0.9]:
    t = f * t_evap
    s = hawking_saddle_entropy(t, M0) + island_saddle_entropy_evaporating(t, M0)
    print(f't/t_evap = {f}:  S_H + S_island = {s:.12f}  (expected {S_BH_0:.12f})')

## 3. The Page time — closed form

The two saddles cross when the BH has lost half its initial entropy: $S_{BH}(t_P) = S_{BH}(0)/2$. Since $S_{BH} = 4\pi M^2$, this means $M(t_P) = M_0/\sqrt{2}$. Substituting into the cubic law:

$$\frac{1}{\sqrt 2} = \left(1 - \frac{t_P}{t_{evap}}\right)^{1/3}
\;\Longrightarrow\;
\boxed{\;\frac{t_P}{t_{evap}} = 1 - \frac{\sqrt{2}}{4} \approx 0.6464\;}$$

The Page time is **not** at $t_{evap}/2$. The BH entropy decreases like $M^2$, so half the entropy is lost much later than half the time.

We verify this two ways: closed-form, and an independent `brentq` on the residual.

In [ ]:
t_P_cf = page_time_evaporating(M0)
t_P_num = page_time_evaporating_numerical(M0)

print(f't_P closed form  = {t_P_cf:.10f}')
print(f't_P numerical    = {t_P_num:.10f}')
print(f'absolute residual = {abs(t_P_cf - t_P_num):.2e}')
print()
print(f't_P / t_evap     = {t_P_cf / t_evap:.10f}')
print(f'1 - sqrt(2)/4    = {1 - math.sqrt(2)/4:.10f}')
print()
print(f'M(t_P)           = {schwarzschild_mass(t_P_cf, M0):.12f}')
print(f'M_0 / sqrt(2)    = {M0 / math.sqrt(2):.12f}')

## 4. The headline plot — the bell-shaped Page curve

$$S_\text{rad}(t) = \min\!\bigl(S_H(t),\, S_\text{island}(t)\bigr)$$

Rising along the Hawking saddle until $t_P$, then falling along the island saddle to zero at $t_{evap}$. Bell-shaped, peak at $S_{BH}(0)/2 = 2\pi M_0^2$, **endpoints exactly at zero**. *That return to zero is unitarity.*

In [ ]:
page_vals = np.array([page_curve_evaporating(t, M0)[0] for t in ts])

fig, ax = plt.subplots(figsize=(10, 6))

# The two saddles, faded as background
ax.plot(ts / t_evap, S_H_arr, lw=1.5, color='#d62728', alpha=0.45,
        label=r'$S_H(t)$  (no-island / Hawking)')
ax.plot(ts / t_evap, S_I_arr, lw=1.5, color='#2ca02c', alpha=0.45,
        label=r'$S_\mathrm{island}(t)$  (QES saddle)')

# The Page curve, bold
ax.plot(ts / t_evap, page_vals, lw=3.0, color='#1f1f1f',
        label=r'$S_\mathrm{rad}(t) = \min(\,\cdot\,, \,\cdot\,)$  — the Page curve')

# Page time marker and peak
ax.axvline(t_P_cf / t_evap, color='gray', ls=':', lw=1.2,
           label=fr'$t_P / t_\mathrm{{evap}} = 1 - \sqrt 2/4 \approx {t_P_cf/t_evap:.4f}$')
ax.axhline(S_BH_0 / 2, color='gray', ls='--', lw=1, alpha=0.5,
           label=r'$S_{BH}(0)/2 = 2\pi M_0^2$  (predicted maximum)')
ax.scatter([t_P_cf / t_evap], [S_BH_0 / 2], s=80, color='black', zorder=5)

ax.set_xlabel(r'$t / t_\mathrm{evap}$', fontsize=13)
ax.set_ylabel(r'entropy', fontsize=13)
ax.set_title('The evaporating-BH Page curve — bell-shaped, returns to zero at $t_\\mathrm{evap}$',
             fontsize=13)
ax.legend(loc='upper right', fontsize=10, framealpha=0.92)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, S_BH_0 * 1.02)

# Annotate endpoints
ax.annotate('$S_\\mathrm{rad}(0) = 0$',
            xy=(0, 0), xytext=(0.05, 1.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=1),
            fontsize=11)
ax.annotate('$S_\\mathrm{rad}(t_\\mathrm{evap}) = 0$\n(unitarity)',
            xy=(1.0, 0), xytext=(0.78, 1.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=1),
            fontsize=11, ha='center')

plt.tight_layout()
plt.show()

## 5. Phase 9 vs v1.1 — visual comparison

Side by side: the eternal BH plateau vs the evaporating bell. Same *idea* (two saddles compete, the min wins, the crossover is the Page time), different *late-time behaviour*. The eternal BH never actually evaporates so the curve has nowhere to fall.

In [ ]:
from spacetime_lab.holography import page_curve, page_time as eternal_page_time

# Reasonable BTZ parameters for the eternal-BH curve
r_plus = 1.0
L = 1.0
eps = 0.01
G_N = 1.0

t_P_eternal = eternal_page_time(r_plus, L, eps, G_N=G_N)
t_max_eternal = 2.5 * t_P_eternal
ts_eternal = np.linspace(0, t_max_eternal, 400)
S_eternal = np.array([page_curve(t, r_plus, L, eps, G_N=G_N)[0] for t in ts_eternal])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Eternal: rises and saturates
axes[0].plot(ts_eternal / t_P_eternal, S_eternal, lw=2.5, color='#1f77b4')
axes[0].axvline(1.0, color='gray', ls=':', lw=1.2)
axes[0].set_xlabel(r'$t / t_P$')
axes[0].set_ylabel('entropy')
axes[0].set_title('Phase 9 — eternal BTZ (saturates)')
axes[0].grid(alpha=0.3)

# Evaporating: rises and falls
axes[1].plot(ts / t_evap, page_vals, lw=2.5, color='#d62728')
axes[1].axvline(t_P_cf / t_evap, color='gray', ls=':', lw=1.2)
axes[1].set_xlabel(r'$t / t_\mathrm{evap}$')
axes[1].set_ylabel('entropy')
axes[1].set_title('v1.1 — evaporating Schwarzschild (bell)')
axes[1].grid(alpha=0.3)

plt.suptitle('Same physics, different background: the late-time tail is the evaporation', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Closing gate — every claim is bit-exact

Same pattern as every previous Spacetime Lab phase: the closing cell of the notebook hard-asserts every closed-form invariant. Failure here means the v1.1 release is broken.

In [ ]:
diag = verify_evaporating_unitarity(M0, n_samples=2000)

print('=== v1.1 evaporating Page curve — closing gate ===')
print()
for k, v in diag.items():
    print(f'  {k:.<35} {v}')
print()

# Hard asserts
assert diag['page_time_residual']    < 1e-10, 'Page time closed-form != numerical'
assert diag['mass_residual']         < 1e-12, 'M(t_P) != M_0/sqrt(2)'
assert diag['continuity_residual']   < 1e-12, 'S_H(t_P) != S_island(t_P)'
assert diag['max_entropy_residual']  < 1e-12, 'S_max != S_BH(0)/2'
assert diag['endpoint_zero_t0']     == 0.0,   'S_rad(0) != 0'
assert diag['endpoint_zero_tevap']  == 0.0,   'S_rad(t_evap) != 0'
assert diag['phase_ordering_correct'],         'phase ordering wrong'
assert diag['bell_shape_monotone'],            'curve not bell-shaped'

print('✓ Every gate passes to machine precision.')
print('✓ The Page curve closes back to zero at t_evap — unitarity in the toy model.')
print('✓ v1.1 is good to ship.')

## What v1.1 ships, and what it doesn't

**Ships:**
- The Schwarzschild evaporation law (Page 1976) implemented with the canonical $5120\pi G^2 M_0^3$ prefactor.
- The bell-shaped Page curve as $\min(S_H, S_\text{island})$ for an evaporating BH.
- A closed-form Page time $t_P = (1 - \sqrt{2}/4)\,t_{evap}$ — no transcendental, no log corrections.
- Bit-exact gate tests on every invariant.
- An end-to-end demonstration of toy-model unitarity: radiation entropy returns exactly to zero at the end of evaporation.

**Deferred to v2.0:**
- The full quantum extremal surface formalism with a real QES finder.
- JT gravity + CFT bath setup (Penington / AEMM original 2D models).
- Replica wormholes — analytical, not coded.
- Backreaction of the radiation on the geometry.
- Grey-body factors and species multiplicity (folded into `t_evap` here).

These are honest scope decisions, documented in the CHANGELOG with the *reason* and the *trigger condition*. The same honest-scope discipline that shipped Phases 3, 4, 5, 6, 8, and 9 without ever shipping a broken-but-deferred feature.